<a href="https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

**Method: Random Forest, compared against Logistic Regression and a depth-3 Decision Tree, all evaluated as ranking scores (Precision@50) rather than as bare classifiers.**

Per the `training-honest-models` skill, method choice follows the question's shape, not habit: this is a *"which ones first?"* ranking problem (named that way already in my `w01`/`w02` notebooks), so any classifier's `predict_proba` output is usable as long as it's evaluated at precision@K, not accuracy. I start simple (logistic regression — fully readable coefficients) and add complexity only if it earns its keep (a depth-3 tree I can print and read, then a random forest) — the same ladder the `building-baselines` and `training-honest-models` skills both push for.

This isn't a blind repeat of my `w01`/`w02` numbers. Those (`baseline 0.240` vs `random forest 0.740` Precision@50) came from the small 30,000-row anonymized starter CSV, with the weaker `trend_direction` proxy label (same-window comparison, not a genuine forward split). This notebook re-earns that comparison on the real warehouse, using the `is_declining_next30` label my `w03_data_contract` (ML-04) notebook built — Q1 2026 features strictly before an April 2026 outcome — and the `content_age_days` staleness signal my `w04_baseline_score` (ML-07) notebook validated. The lane guide is explicit that the starter-CSV result "has to be earned all over again, with proper validation" at warehouse scale — that's this notebook's job, not an assumption.


In [6]:
%pip -q install duckdb

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Same Q1 2026 -> April 2026 windows as ML-04 (data contract) and ML-07 (baseline) --
# this notebook is self-contained (re-pulls its own frame), same convention as w03/w04.
FEATURE_START = "DATE '2026-01-01'"
FEATURE_END   = "DATE '2026-03-31'"
LABEL_START   = "DATE '2026-04-01'"
LABEL_END     = "DATE '2026-04-30'"

frame = con.sql(f"""
    WITH eligible_clients AS (
        SELECT client_hash_id
        FROM {TABLES['dim_clients']}
        WHERE gsc_data_start <= {FEATURE_START}
    ),
    feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions)                                                                AS imp_90d,
               AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END)                      AS pos_90d,
               SUM(CASE WHEN f.report_date >  {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_first60,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)                 AS days_with_impressions_90d,
               SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.sessions_ai ELSE 0 END)              AS ai_sessions_90d,
               SUM(f.gsc_clicks)                                                                      AS clicks_90d
        FROM {FACT} f
        JOIN eligible_clients c USING (client_hash_id)
        WHERE f.report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
        GROUP BY 1, 2
        HAVING imp_90d >= 100
    ),
    label AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_label30
        FROM {FACT}
        WHERE report_date BETWEEN {LABEL_START} AND {LABEL_END}
        GROUP BY 1, 2
    ),
    content_age AS (
        SELECT content_hash_id,
               DATE_DIFF('day', CAST(content_created_date AS DATE), {FEATURE_END}) AS content_age_days
        FROM {TABLES['dim_content']}
    )
    SELECT f.*,
           COALESCE(l.imp_label30, 0)                                             AS imp_label30,
           f.imp_last30 / NULLIF(f.imp_first60 / 2.0, 0)                          AS trend_ratio_90d,
           f.ai_sessions_90d / NULLIF(f.clicks_90d, 0)                            AS ai_referral_share_90d,
           CASE WHEN COALESCE(l.imp_label30, 0) < 0.8 * f.imp_last30 THEN 1 ELSE 0 END AS is_declining_next30,
           ca.content_age_days
    FROM feat f
    LEFT JOIN label l USING (client_hash_id, content_hash_id)
    LEFT JOIN content_age ca USING (content_hash_id)
""").df()

frame = frame.dropna(subset=['content_age_days']).copy()

# Baseline score, frozen exactly as ML-07 (w04_baseline_score) built it -- never refit here.
STALE_DAYS, VISIBLE_IMP = 180, 500
frame['stale'] = (frame['content_age_days'] >= STALE_DAYS).astype(int)
frame['visible'] = (frame['imp_90d'] >= VISIBLE_IMP).astype(int)
frame['declining_now'] = (frame['trend_ratio_90d'] < 0.8).astype(int)
frame['baseline_score'] = frame['stale'] * frame['visible'] * frame['imp_90d'] * (1 + frame['declining_now'])

FEATURES = ['imp_90d', 'pos_90d', 'trend_ratio_90d', 'days_with_impressions_90d',
            'ai_referral_share_90d', 'content_age_days']

print(f"{len(frame):,} rows, {frame['client_hash_id'].nunique()} clients")
print("base rate (is_declining_next30):", round(frame['is_declining_next30'].mean(), 3))
print("\nmissing values per feature (before impute):")
print(frame[FEATURES].isna().sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

102,769 rows, 30 clients
base rate (is_declining_next30): 0.494

missing values per feature (before impute):
imp_90d                          0
pos_90d                          0
trend_ratio_90d               9131
days_with_impressions_90d        0
ai_referral_share_90d        33568
content_age_days                 0
dtype: int64


## 2. Split design

**Grouped by client (`client_hash_id`), 80/20, fixed seed — not a random row split.**

Per the `hunting-leakage-and-validating` skill: rows from the same client share hidden character (the same site, the same editorial patterns, the same baseline traffic level), so a random row split lets a model partly memorize the client rather than learn a generalizable signal. The honest question is *"does this rank well on a client it has never seen?"* — the same idea `GUIDE.md` names directly ("hold out ~20% of clients... use the same idea when you evaluate your own models") and the same `client_holdout` strategy the reference pipeline's own `outputs/model_report.md` reports against.

I compute **both** a grouped split and a naive random row-level split on the same data, same seed, same test proportion — and report both in the next section. The skill is explicit that the *gap* between them is itself a finding about how much memorization was happening, not just a formality to satisfy.


In [7]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

SEED = 42
TEST_SIZE = 0.20

# Grouped split -- the honest one.
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(gss.split(frame, groups=frame['client_hash_id']))
train_g, test_g = frame.iloc[train_idx].copy(), frame.iloc[test_idx].copy()

overlap = set(train_g['client_hash_id']) & set(test_g['client_hash_id'])
assert not overlap, f"grouped split leaked {len(overlap)} clients into both sides"

print("GROUPED split (by client_hash_id):")
print(f"  train: {len(train_g):,} rows, {train_g['client_hash_id'].nunique()} clients, "
      f"label rate {train_g['is_declining_next30'].mean():.3f}")
print(f"  test:  {len(test_g):,} rows, {test_g['client_hash_id'].nunique()} clients, "
      f"label rate {test_g['is_declining_next30'].mean():.3f}")
print(f"  client overlap between train and test: {len(overlap)} (must be 0)")

# Naive random row split -- for comparison ONLY, never used to report a headline number.
train_r, test_r = train_test_split(frame, test_size=TEST_SIZE, random_state=SEED)
shared_clients = set(train_r['client_hash_id']) & set(test_r['client_hash_id'])
print("\nNAIVE random row split (comparison only, not the honest number):")
print(f"  train: {len(train_r):,} rows   test: {len(test_r):,} rows")
print(f"  clients appearing on BOTH sides: {len(shared_clients)} of "
      f"{frame['client_hash_id'].nunique()} total -- this is exactly the memorization risk "
      "the grouped split above removes.")


GROUPED split (by client_hash_id):
  train: 91,036 rows, 24 clients, label rate 0.504
  test:  11,733 rows, 6 clients, label rate 0.414
  client overlap between train and test: 0 (must be 0)

NAIVE random row split (comparison only, not the honest number):
  train: 82,215 rows   test: 20,554 rows
  clients appearing on BOTH sides: 29 of 30 total -- this is exactly the memorization risk the grouped split above removes.


## 3. Train + compare vs my baseline

Same feature set (section 1), same grouped test split (section 2), same metric family as my `w01`/`w02` notebooks — Precision@50 as the primary ranking metric (the number tied to the real decision), ROC AUC and average precision as classifier diagnostics, base rate printed next to every score per the `hunting-leakage-and-validating` skill.

The baseline row in the table below is the **frozen** ML-07 rule score, evaluated on the exact same grouped test rows the models are scored on — not refit, not re-tuned. Per `building-baselines`: "keep the baseline frozen once the model work starts."


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

imputer = SimpleImputer(strategy='median').fit(train_g[FEATURES])
X_train = imputer.transform(train_g[FEATURES])
X_test  = imputer.transform(test_g[FEATURES])
y_train = train_g['is_declining_next30'].values
y_test  = test_g['is_declining_next30'].values

models = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'decision_tree':       DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'random_forest':       RandomForestClassifier(n_estimators=300, random_state=SEED),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    proba = m.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {
        'roc_auc': roc_auc_score(y_test, proba),
        'avg_precision': average_precision_score(y_test, proba),
        'precision_at_50': precision_at_k(proba, y_test, 50),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
    }

# Baseline, frozen, scored on the SAME grouped test rows.
base_scores = test_g['baseline_score'].values
results['baseline_rules'] = {
    'roc_auc': roc_auc_score(y_test, base_scores),
    'avg_precision': average_precision_score(y_test, base_scores),
    'precision_at_50': precision_at_k(base_scores, y_test, 50),
    'recall': float('nan'),
    'f1': float('nan'),
}

base_rate = y_test.mean()
print(f"base rate on grouped test split (majority class if always predicting positive): {base_rate:.3f}\n")
print(f"{'model':<22}{'ROC AUC':>10}{'avg precision':>16}{'Precision@50':>16}{'recall':>10}{'F1':>8}")
for name, r in results.items():
    print(f"{name:<22}{r['roc_auc']:>10.3f}{r['avg_precision']:>16.3f}{r['precision_at_50']:>16.3f}"
          f"{r['recall']:>10.3f}{r['f1']:>8.3f}")

best_name = max((n for n in results if n != 'baseline_rules'), key=lambda n: results[n]['precision_at_50'])
lift = results[best_name]['precision_at_50'] - results['baseline_rules']['precision_at_50']
print(f"\nBest model by Precision@50 on the grouped, held-out client split: {best_name} "
      f"({results[best_name]['precision_at_50']:.3f} vs baseline {results['baseline_rules']['precision_at_50']:.3f}, "
      f"a {lift:+.3f} difference).")
if results[best_name]['precision_at_50'] <= results['baseline_rules']['precision_at_50']:
    print("The model did NOT beat the frozen baseline on unseen clients -- that is a valid, reportable result,")
    print("not a failed cell. Section 4 below looks at why, and the paper's Evaluation section says so plainly.")

# --- Cross-check: same models, same features, scored under the NAIVE split from section 2 ---
# This is the "before/after" honesty gap (also reused directly in ML-09, w06_validation_audit).
X_train_r = imputer.transform(train_r[FEATURES])
X_test_r  = imputer.transform(test_r[FEATURES])
y_train_r = train_r['is_declining_next30'].values
y_test_r  = test_r['is_declining_next30'].values

rf_naive = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(X_train_r, y_train_r)
proba_naive = rf_naive.predict_proba(X_test_r)[:, 1]
p50_naive = precision_at_k(proba_naive, y_test_r, 50)
p50_grouped = results['random_forest']['precision_at_50']
print(f"\nRandom forest Precision@50 -- naive random split: {p50_naive:.3f}  vs  grouped client split: {p50_grouped:.3f}")
gap = p50_naive - p50_grouped
print(f"Gap: {gap:+.3f}. " + ("A positive gap this size is exactly the memorization risk the grouped split "
      "exists to catch -- the naive number is not the one I report as my headline result."
      if gap > 0.03 else
      "The two splits are close here, which is itself worth noting -- it suggests the signal isn't just "
      "client memorization, but the grouped number is still the one I trust and report."))


base rate on grouped test split (majority class if always predicting positive): 0.414

model                    ROC AUC   avg precision    Precision@50    recall      F1
logistic_regression        0.481           0.432           0.600     0.588   0.477
decision_tree              0.570           0.454           0.900     0.506   0.497
random_forest              0.537           0.426           0.440     0.757   0.549
baseline_rules             0.400           0.359           0.340       nan     nan

Best model by Precision@50 on the grouped, held-out client split: decision_tree (0.900 vs baseline 0.340, a +0.560 difference).

Random forest Precision@50 -- naive random split: 0.980  vs  grouped client split: 0.440
Gap: +0.540. A positive gap this size is exactly the memorization risk the grouped split exists to catch -- the naive number is not the one I report as my headline result.


## 4. Errors and interpretation

Feature importances from the random forest, sanity-checked against the `hunting-leakage-and-validating` skill's warning: a single feature towering over the rest with a near-perfect score is the "suspiciously perfect" pattern that usually means leakage, not skill. Then three concrete wrong cases at the top of the ranked queue (false positives — flagged high, actually stable) and three concrete missed cases (real decliners the model ranked low), read by hand rather than left as a metric.

**Update, after actually running this cell once:** `trend_ratio_90d` came back carrying 68.4% of the decision tree's importance — well past the 60% smell-test threshold above. Per the leakage skill's own verification method ("train once WITH the suspect, once WITHOUT — a collapse is the confession"), the code below now runs that test directly: retrains the top two models without `trend_ratio_90d` and prints both numbers side by side, plus how many distinct scores the decision tree actually produces (a depth-3 tree has at most 8 leaves, so Precision@50 on a small test set can be more tie-breaking than ranking).


In [9]:
best_model = models[best_name] if best_name in models else None

if best_model is not None and hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print(f"Feature importances -- {best_name}:")
    print(importances.round(4))
    top_share = importances.iloc[0]
    print(f"\ntop feature ({importances.index[0]}) carries {top_share:.1%} of total importance.")
    if top_share > 0.60:
        print("That's high enough to treat as a leakage smell, not a win -- worth re-checking that feature")
        print("against the excluded-fields list in ML-04/ML-05 before trusting this model.")
    else:
        print("No single feature dominates -- consistent with a model using several signals together,")
        print("not quietly re-deriving the label from one column.")
else:
    print(f"Best model ({best_name}) has no feature_importances_ (e.g. logistic regression) --")
    print("reporting standardized coefficients instead would be the equivalent check for that model.")
    if best_name == 'logistic_regression':
        coefs = pd.Series(best_model_lr.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False) \
            if 'best_model_lr' in dir() else None

# Rank the grouped test split by the best model's probability, examine the top 50.
test_g = test_g.copy()
test_g['model_proba'] = models[best_name].predict_proba(imputer.transform(test_g[FEATURES]))[:, 1] \
    if best_name in models else base_scores
ranked = test_g.sort_values('model_proba' if best_name in models else 'baseline_score', ascending=False)
top50 = ranked.head(50)

false_positives = top50[top50['is_declining_next30'] == 0]
print(f"\nOf the top 50 ranked test rows, {len(false_positives)} were false positives (flagged, not actually declining).")
show_cols = ['imp_90d', 'pos_90d', 'trend_ratio_90d', 'content_age_days', 'ai_referral_share_90d']
if len(false_positives):
    print("\n3 concrete false-positive examples (flagged high, stayed stable):")
    print(false_positives[show_cols].head(3).round(3))
    print("why these are hard: high on the same visible/stale/declining-now signals the model leans on,")
    print("but April demand held up anyway -- exactly the seasonality/consolidation confound section 7")
    print("of the lane guide warns about, which this model has no way to see from Q1 data alone.")

missed = ranked[(ranked['is_declining_next30'] == 1)].tail(len(ranked[ranked['is_declining_next30'] == 1]) // 2)
low_ranked_decliners = ranked[ranked['is_declining_next30'] == 1].sort_values(
    'model_proba' if best_name in models else 'baseline_score'
).head(3)
if len(low_ranked_decliners):
    print("\n3 concrete false-negative examples (real April decliners the model ranked near the bottom):")
    print(low_ranked_decliners[show_cols].round(3))
    print("why these are hard: quiet on every Q1 signal available here (low volume and/or flat trend_ratio),")
    print("so nothing in this feature set gave the model a reason to flag them before the drop happened.")

# --- Robustness check, run for real: does trend_ratio_90d's apparent edge survive removing it? ---
# It shares imp_last30 with the label's own reference point (is_declining_next30 is defined relative
# to imp_last30), so a page with an unusually high recent rate faces a proportionally higher bar to
# avoid being called "declining" -- a structural coupling with the label, not necessarily new signal.
FEATURES_NO_TREND = [f for f in FEATURES if f != 'trend_ratio_90d']
imputer_nt = SimpleImputer(strategy='median').fit(train_g[FEATURES_NO_TREND])
X_train_nt = imputer_nt.transform(train_g[FEATURES_NO_TREND])
X_test_nt  = imputer_nt.transform(test_g[FEATURES_NO_TREND])

print("\nRobustness check -- Precision@50 WITH vs WITHOUT trend_ratio_90d:")
for name, cls in [('decision_tree', DecisionTreeClassifier(max_depth=3, random_state=SEED)),
                   ('random_forest', RandomForestClassifier(n_estimators=300, random_state=SEED))]:
    m = cls.fit(X_train_nt, y_train)
    proba_nt = m.predict_proba(X_test_nt)[:, 1]
    p50_nt = precision_at_k(proba_nt, y_test, 50)
    print(f"  {name:<16} WITHOUT: {p50_nt:.3f}   WITH: {results[name]['precision_at_50']:.3f}")

dt_proba_test = models['decision_tree'].predict_proba(X_test)[:, 1]
n_distinct = len(set(dt_proba_test))
print(f"\ndistinct decision-tree probability values on the test set: {n_distinct}")
print("(a depth-3 tree has at most 8 leaves -- a small count here means the 'top 50' is largely")
print("tie-broken by row order within one leaf, not a fine-grained ranking.)")

print("\nConclusion: if the WITHOUT number collapses toward the frozen baseline's own Precision@50")
print("(see section 3's results['baseline_rules']), that confirms trend_ratio_90d's apparent power was")
print("mostly this structural coupling, not genuine forward-looking signal. Per that finding, downstream")
print("notebooks (w06_validation_audit, w07_action_playbook, capstone) exclude trend_ratio_90d from")
print("FEATURES going forward -- it stays in the dataframe only for the baseline's declining_now flag.")


Feature importances -- decision_tree:
trend_ratio_90d              0.6837
days_with_impressions_90d    0.1723
content_age_days             0.1440
imp_90d                      0.0000
pos_90d                      0.0000
ai_referral_share_90d        0.0000
dtype: float64

top feature (trend_ratio_90d) carries 68.4% of total importance.
That's high enough to treat as a leakage smell, not a win -- worth re-checking that feature
against the excluded-fields list in ML-04/ML-05 before trusting this model.

Of the top 50 ranked test rows, 7 were false positives (flagged, not actually declining).

3 concrete false-positive examples (flagged high, stayed stable):
        imp_90d  pos_90d  trend_ratio_90d  content_age_days  \
945       178.0    8.582            2.188               104   
102762    124.0   20.029            0.234               214   
83928     502.0   17.723            2.218               230   

        ai_referral_share_90d  
945                       NaN  
102762                

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
